# 04 — Hamiltonian Simulation: Small Cases

**Trilha:** Tradução de problemas  
**Milestone:** 3 — Tradução de problemas clássicos para operadores

---

## Pergunta

Como traduzir um problema físico real (dinâmica de um sistema quântico) para a linguagem de operadores e circuitos? O que é preservado e o que se perde na aproximação?

Hamiltonian simulation é o caso de uso mais natural — e historicamente o mais honesto — de computação quântica. Aqui o problema *já é* quântico. O desafio é implementar $e^{-iHt}$ eficientemente.

## Modelo matemático mínimo

### O problema
Dado um Hamiltoniano $H = H_1 + H_2 + ...$, simular a evolução $U(t) = e^{-iHt}$.

**Dificuldade clássica:** $e^{-iHt}$ é uma matriz $2^n \times 2^n$ para $n$ qubits — exponencialmente grande.

### Decomposição de Trotter
Se $H = A + B$ e $[A, B] \neq 0$ (não comutam):
$$e^{-i(A+B)t} \neq e^{-iAt} e^{-iBt}$$

Mas por primeiro ordem de Trotter:
$$e^{-i(A+B)t} \approx \left(e^{-iAt/r} e^{-iBt/r}\right)^r + O(t^2/r)$$

Erro de Trotter $\sim \|[A,B]\| t^2 / r$. Mais passos $r$ → menor erro → mais portas.

### Por que isso é uma "tradução"
O problema (evolução sob $H$) vira:
- **Operador:** $U(t) = e^{-iHt}$ (unitário)
- **Preparação:** estado inicial $|\psi_0\rangle$
- **Medição:** valor esperado de um observável $O$ em tempo $t$

A decomposição de Trotter é o custo da tradução — ela introduz erro controlável mas não zero.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

# --- Caso mínimo: Hamiltoniano de 2 qubits ---
# H = Z⊗I + I⊗Z + g·X⊗X  (campo transverso 1D, 2 sítios)

# Matrizes de Pauli
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# Hamiltoniano de 2 qubits
g = 0.5  # acoplamento
H_ZI = np.kron(Z, I)
H_IZ = np.kron(I, Z)
H_XX = np.kron(X, X)
H = H_ZI + H_IZ + g * H_XX

print('Hamiltoniano H = Z⊗I + I⊗Z + 0.5·X⊗X')
print('Dimensão:', H.shape)
print()

# Espectro exato
eigvals, eigvecs = np.linalg.eigh(H)
print('Autovalores (energias):', eigvals.round(4))
print('Ground state energy:', eigvals[0].round(6))

In [ ]:
# --- Trotter vs. evolução exata ---

def trotter_step(H_A, H_B, t, r):
    """Aproximação de Trotter de 1ª ordem: (exp(-iA·t/r) exp(-iB·t/r))^r"""
    step_A = expm(-1j * H_A * t / r)
    step_B = expm(-1j * H_B * t / r)
    single_step = step_B @ step_A  # B depois de A
    return np.linalg.matrix_power(single_step, r)

# Dividir H = (Z⊗I + I⊗Z) + g·(X⊗X)
H_free = H_ZI + H_IZ  # parte livre
H_int  = g * H_XX     # interação

# Estado inicial: |00>
psi0 = np.zeros(4, dtype=complex)
psi0[0] = 1.0  # |00>

times = np.linspace(0, 5.0, 100)
r_values = [1, 4, 16]

# Valor esperado de Z⊗I
obs = H_ZI

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Evolução exata
exact_vals = []
for t in times:
    U_exact = expm(-1j * H * t)
    psi_t = U_exact @ psi0
    exact_vals.append(np.real(psi_t.conj() @ obs @ psi_t))

axes[0].plot(times, exact_vals, 'k-', label='Exato', linewidth=2)

colors = ['steelblue', 'coral', 'green']
for r, color in zip(r_values, colors):
    trotter_vals = []
    for t in times:
        U_trotter = trotter_step(H_free, H_int, t, r)
        psi_t = U_trotter @ psi0
        trotter_vals.append(np.real(psi_t.conj() @ obs @ psi_t))
    axes[0].plot(times, trotter_vals, '--', color=color, label=f'Trotter r={r}', alpha=0.8)

axes[0].set_xlabel('Tempo t')
axes[0].set_ylabel('⟨Z⊗I⟩')
axes[0].set_title('Dinâmica: Trotter vs. Exato')
axes[0].legend()

# Erro de Trotter vs. r
t_fixed = 2.0
U_exact_fixed = expm(-1j * H * t_fixed)
r_range = [1, 2, 4, 8, 16, 32, 64]
errors = []
for r in r_range:
    U_t = trotter_step(H_free, H_int, t_fixed, r)
    errors.append(np.linalg.norm(U_t - U_exact_fixed))

axes[1].loglog(r_range, errors, 'o-', color='steelblue')
# Linha de referência O(1/r)
r_ref = np.array(r_range, dtype=float)
axes[1].loglog(r_ref, errors[0] * r_range[0] / r_ref, 'k--', alpha=0.5, label='O(1/r)')
axes[1].set_xlabel('Número de passos r')
axes[1].set_ylabel('||U_Trotter - U_exato||')
axes[1].set_title(f'Erro de Trotter em t={t_fixed}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Relação com QPE: estimar energias via Hamiltonian simulation ---

# Se quisermos as energias de H via QPE:
# 1. Implementar U(t) = e^(-iHt) como circuito (via Trotter)
# 2. Rodar QPE sobre U(t) com autovetor de H
# 3. QPE retorna φ tal que e^(2πiφ) = autovalor de U(t) = e^(-iE·t)
# => E = -2π·φ / t

# Simulação direta (sem QPE completo): verificar a relação
t_qpe = 1.0  # tempo de evolução
U_t = expm(-1j * H * t_qpe)

# Autovalores de U(t)
eigvals_U = np.linalg.eigvals(U_t)
phases = np.angle(eigvals_U)  # fases de U(t)
energies_recovered = -phases / t_qpe  # E = -fase / t

print('Relação QPE ↔ Hamiltonian simulation:')
print(f'\n{"Energia (exata)":20} {"Energia (via fase de U):"}')
for E_exact, E_rec in zip(sorted(eigvals), sorted(energies_recovered)):
    print(f'  {E_exact:8.4f}            {E_rec:8.4f}')

print()
print('QPE sobre U(t) = e^(-iHt) recupera autovalores de H via suas fases.')
print('Hamiltonian simulation é o passo que torna esse circuito implementável.')

## Conclusão

1. **Hamiltonian simulation é a tradução mais natural.** O problema *já é* quântico: calcular $e^{-iHt}|\psi_0\rangle$. A tradução para circuito é quase direta — o custo está na decomposição.

2. **Decomposição de Trotter introduz erro controlável.** O erro de 1ª ordem é $O(t^2/r)$. Mais passos $r$ → melhor precisão → mais portas. Isso é o custo da tradução.

3. **QPE sobre $e^{-iHt}$ recupera autovalores de $H$.** Essa é a cadeia completa: Hamiltoniano → simulação → phase estimation → energias. Cada elo tem seu custo.

4. **O que foi preservado na tradução:** a dinâmica exata (no limite $r → \infty$) e a estrutura espectral de $H$.

5. **O que se perde:** a exatidão (Trotter é uma aproximação), e a eficiência pode se perder para Hamiltonianos densos sem estrutura esparsa.

---

**Próximo:** problema explicitamente clássico sendo traduzido — operador, estado inicial, medição — com análise do custo de preparação.